In [1]:
from torch_geometric.datasets import EllipticBitcoinDataset

dataset = EllipticBitcoinDataset(root='/tmp/Elliptic')
data = dataset[0]

print(f"Nodes: {data.num_nodes}")       # 203769
print(f"Edges: {data.num_edges}")       # 234355
print(f"Features: {data.num_features}") # 165
# Note: only ~46K nodes are labeled (data.y != -1)
labeled = (data.y >= 0).sum()
print(f"Labeled nodes: {labeled}")

Processing...


Nodes: 203769
Edges: 234355
Features: 165
Labeled nodes: 203769


Done!


In [10]:
for d in data:
    print(d)

('x', tensor([[-0.1715, -0.1847, -1.2014,  ..., -0.0975, -0.1206, -0.1198],
        [-0.1715, -0.1847, -1.2014,  ..., -0.0975, -0.1206, -0.1198],
        [-0.1721, -0.1847, -1.2014,  ..., -0.1837, -0.1206, -0.1198],
        ...,
        [-0.1720, -0.0782,  1.0186,  ..., -0.0975, -0.1206, -0.1198],
        [-0.1728, -0.1766,  1.0186,  ..., -0.1406,  1.5197,  1.5214],
        [-0.0120, -0.1323,  0.4636,  ..., -0.1406,  1.5197,  1.5214]]))
('edge_index', tensor([[     0,      2,      4,  ..., 201921, 201480, 201954],
        [     1,      3,      5,  ..., 202042, 201368, 201756]]))
('y', tensor([2, 2, 2,  ..., 1, 2, 2]))
('train_mask', tensor([False, False, False,  ..., False, False, False]))
('test_mask', tensor([False, False, False,  ...,  True, False, False]))


In [28]:
import pandas as pd

labels =pd.read_csv("/home/yeli/Desktop/phd/cjepa/dataset/elliptic_bitcoin/elliptic_txs_classes.csv")
features = pd.read_csv("/home/yeli/Desktop/phd/cjepa/dataset/elliptic_bitcoin/elliptic_txs_features.csv", header=None)
edges = pd.read_csv("/home/yeli/Desktop/phd/cjepa/dataset/elliptic_bitcoin/elliptic_txs_edgelist.csv")

T = features[1].max()
assert len(labels) == len(features)
N = len(labels)
features

,0,1,2,3,4,5,6,7,8,9,...,157,158,159,160,161,162,163,164,165,166
0,230425980,1,-0.171469,-0.184668,-1.201369,-0.121970,-0.043875,-0.113002,-0.061584,-0.162097,...,-0.562153,-0.600999,1.461330,1.461369,0.018279,-0.087490,-0.131155,-0.097524,-0.120613,-0.119792
1,5530458,1,-0.171484,-0.184668,-1.201369,-0.121970,-0.043875,-0.113002,-0.061584,-0.162112,...,0.947382,0.673103,-0.979074,-0.978556,0.018279,-0.087490,-0.131155,-0.097524,-0.120613,-0.119792
2,232022460,1,-0.172107,-0.184668,-1.201369,-0.121970,-0.043875,-0.113002,-0.061584,-0.162749,...,0.670883,0.439728,-0.979074,-0.978556,-0.098889,-0.106715,-0.131155,-0.183671,-0.120613,-0.119792
3,232438397,1,0.163054,1.963790,-0.646376,12.409294,-0.063725,9.782742,12.414558,-0.163645,...,-0.577099,-0.613614,0.241128,0.241406,1.072793,0.085530,-0.131155,0.677799,-0.120613,-0.119792
4,230460314,1,1.011523,-0.081127,-1.201369,1.153668,0.333276,1.312656,-0.061584,-0.163523,...,-0.511871,-0.400422,0.517257,0.579382,0.018279,0.277775,0.326394,1.293750,0.178136,0.179117
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
203764,173077460,49,-0.145771,-0.163752,0.463609,-0.121970,-0.043875,-0.113002,-0.061584,-0.135803,...,-0.577099,-0.613614,0.241128,0.241406,0.018279,-0.087490,-0.131155,-0.097524,-0.120613,-0.119792
203765,158577750,49,-0.165920,-0.123607,1.018602,-0.121970,-0.043875,-0.113002,-0.061584,-0.156418,...,0.162722,0.010822,1.461330,1.461369,-0.098889,-0.087490,-0.084674,-0.140597,-1.760926,-1.760984
203766,158375402,49,-0.172014,-0.078182,1.018602,0.028105,-0.043875,0.054722,-0.061584,-0.163626,...,1.261246,1.985050,1.461330,1.461369,0.018279,-0.087490,-0.131155,-0.097524,-0.120613,-0.119792
203767,158654197,49,-0.172842,-0.176622,1.018602,-0.121970,-0.043875,-0.113002,-0.061584,-0.163501,...,-0.397749,-0.411776,1.461330,1.461369,-0.098889,-0.087490,-0.084674,-0.140597,1.519700,1.521399


In [ ]:
import numpy as np
import scipy.sparse as sp

# txId -> 0..N-1 (row order of the features file defines the canonical index)
node_ids = features[0].to_numpy()
id_to_idx = {tx: i for i, tx in enumerate(node_ids)}
N = len(node_ids)

src_idx = np.fromiter((id_to_idx[x] for x in edges['txId1'].to_numpy()), dtype=np.int64)
dst_idx = np.fromiter((id_to_idx[x] for x in edges['txId2'].to_numpy()), dtype=np.int64)

# timestep per edge (via source; asserted equal to dst below)
ts = features.set_index(0)[1]
t_src = ts.loc[edges['txId1'].to_numpy()].to_numpy()
t_dst = ts.loc[edges['txId2'].to_numpy()].to_numpy()
assert (t_src == t_dst).all(), "cross-timestep edge found"
t_edge = t_src

adjs = []
for t in range(1, T + 1):                 # Elliptic timesteps are 1..49
    m = t_edge == t                       # edge-level mask
    A = sp.csr_matrix(
        (np.ones(m.sum(), dtype=np.float32), (src_idx[m], dst_idx[m])),
        shape=(N, N),
    )
    adjs.append(A)                        # inside the loop

IndexError: boolean index did not match indexed array along axis 0; size of axis is 234355 but size of corresponding boolean axis is 203769

,txId,class
0,230425980,unknown
1,5530458,unknown
2,232022460,unknown
3,232438397,2
4,230460314,unknown
...,...,...
203764,173077460,unknown
203765,158577750,unknown
203766,158375402,1
203767,158654197,unknown


In [25]:
features[1].max()

np.int64(49)

In [21]:
features[features[0]==158589457]

,0,1,2,3,4,5,6,7,8,9,...,157,158,159,160,161,162,163,164,165,166
201756,158589457,49,-0.172083,-0.141522,1.018602,-0.12197,-0.043875,-0.113002,-0.061584,-0.162723,...,-0.128722,-0.235168,1.46133,1.461369,0.018279,-0.08749,-0.131155,-0.097524,-0.120613,-0.119792


In [8]:
pd.read_csv("/home/yeli/Desktop/phd/cjepa/dataset/elliptic_bitcoin/elliptic_txs_features.csv", header=None).head()

,0,1,2,3,4,5,6,7,8,9,...,157,158,159,160,161,162,163,164,165,166
0,230425980,1,-0.171469,-0.184668,-1.201369,-0.121970,-0.043875,-0.113002,-0.061584,-0.162097,...,-0.562153,-0.600999,1.461330,1.461369,0.018279,-0.087490,-0.131155,-0.097524,-0.120613,-0.119792
1,5530458,1,-0.171484,-0.184668,-1.201369,-0.121970,-0.043875,-0.113002,-0.061584,-0.162112,...,0.947382,0.673103,-0.979074,-0.978556,0.018279,-0.087490,-0.131155,-0.097524,-0.120613,-0.119792
2,232022460,1,-0.172107,-0.184668,-1.201369,-0.121970,-0.043875,-0.113002,-0.061584,-0.162749,...,0.670883,0.439728,-0.979074,-0.978556,-0.098889,-0.106715,-0.131155,-0.183671,-0.120613,-0.119792
3,232438397,1,0.163054,1.963790,-0.646376,12.409294,-0.063725,9.782742,12.414558,-0.163645,...,-0.577099,-0.613614,0.241128,0.241406,1.072793,0.085530,-0.131155,0.677799,-0.120613,-0.119792
4,230460314,1,1.011523,-0.081127,-1.201369,1.153668,0.333276,1.312656,-0.061584,-0.163523,...,-0.511871,-0.400422,0.517257,0.579382,0.018279,0.277775,0.326394,1.293750,0.178136,0.179117


In [3]:
# Access the first node
data.x[0]

tensor([-1.7147e-01, -1.8467e-01, -1.2014e+00, -1.2197e-01, -4.3875e-02,
        -1.1300e-01, -6.1584e-02, -1.6210e-01, -1.6793e-01, -4.9707e-02,
        -1.6440e-01, -2.8741e-02, -3.5391e-02, -4.2955e-02, -1.3282e-02,
        -5.7195e-02, -1.6961e-01, -1.7115e-01, -1.7447e-01, -1.3737e+00,
        -1.3715e+00, -1.3973e-01, -1.4891e-01, -8.0147e-02, -1.5566e-01,
        -1.0763e-02, -1.2107e-02, -1.3973e-01, -1.4891e-01, -8.0147e-02,
        -1.5566e-01, -1.0669e-02, -1.2005e-02, -2.4669e-02, -3.1272e-02,
        -2.3045e-02, -2.6215e-02,  1.4278e-03,  1.4826e-03, -2.2722e-01,
        -2.3937e-01, -7.5256e-02, -2.3495e-01,  3.7468e-02,  4.3444e-02,
        -2.2720e-01, -2.4324e-01, -9.7895e-02, -2.3590e-01,  3.6577e-02,
         4.2345e-02, -4.1401e-01, -4.8834e-01, -2.3255e-01, -4.6755e-01,
         4.8767e-02,  5.2956e-02, -3.9149e-02, -1.7290e-01, -1.6313e-01,
        -1.6093e-01, -1.3163e+00, -1.3154e+00, -3.9144e-02, -1.7288e-01,
        -1.6311e-01, -1.6093e-01, -1.3163e+00, -1.3